# 02 — Análise Exploratória dos Dados

## Objetivo

Realizar a análise exploratória da base analítica da RAIS 2025 para o
estado de São Paulo, investigando a associação entre escolaridade e
remuneração e verificando como essa relação varia entre os setores de
atividade econômica.

A análise busca responder principalmente às seguintes perguntas:

1. Como os vínculos formais estão distribuídos entre os níveis de escolaridade?
2. Como a remuneração varia conforme o nível de escolaridade?
3. Como a associação entre escolaridade e remuneração varia entre os setores econômicos?

A unidade de análise é o vínculo formal de trabalho.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [9]:
DATA_PROCESSED = Path("../data/processed")
FIGURES = Path("../figures")

arquivo = DATA_PROCESSED / "rais_sp_2025.parquet"

FIGURES.mkdir(exist_ok=True)

In [10]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(arquivo)

df = parquet_file.read().to_pandas()

FileNotFoundError: [WinError 2] Failed to open local file '../data/processed/rais_sp_2025.parquet'. Detail: [Windows error 2] O sistema não pode encontrar o arquivo especificado.


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
ordem_escolaridade = [
    "Analfabeto",
    "Até 5º ano incompleto",
    "5º ano completo do Fundamental",
    "6º ao 9º ano do Fundamental",
    "Fundamental completo",
    "Médio incompleto",
    "Médio completo",
    "Superior incompleto",
    "Superior completo",
    "Mestrado",
    "Doutorado"
]

In [ ]:
df["Escolaridade"] = pd.Categorical(
    df["Escolaridade"],
    categories=ordem_escolaridade,
    ordered=True
)

## 1. Distribuição dos vínculos por escolaridade

A primeira análise busca identificar como os vínculos formais da
população analítica estão distribuídos entre os diferentes níveis de
escolaridade.

In [ ]:
distribuicao_escolaridade = (
    df["Escolaridade"]
    .value_counts(sort=False)
    .rename("Vínculos")
    .to_frame()
)

distribuicao_escolaridade["Percentual"] = (
    distribuicao_escolaridade["Vínculos"]
    / distribuicao_escolaridade["Vínculos"].sum()
    * 100
)

distribuicao_escolaridade

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    distribuicao_escolaridade.index.astype(str),
    distribuicao_escolaridade["Percentual"]
)

ax.set_xlabel("Percentual dos vínculos (%)")
ax.set_ylabel("Escolaridade")
ax.set_title(
    "Distribuição dos vínculos formais por nível de escolaridade\n"
    "São Paulo — RAIS 2025"
)

ax.invert_yaxis()

plt.tight_layout()

plt.savefig(
    FIGURES / "distribuicao_escolaridade.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Interpretação

A distribuição dos vínculos apresenta forte concentração nos níveis
Médio completo e Superior completo. O Ensino Médio completo representa
aproximadamente 52% da população analítica, enquanto o Superior completo
representa cerca de 29%.

Em conjunto, essas duas categorias correspondem a aproximadamente 81%
dos vínculos analisados. Os demais níveis de escolaridade apresentam
participações consideravelmente menores.

In [ ]:
remuneracao_escolaridade = (
    df.groupby(
        "Escolaridade",
        observed=True
    )["Vl Rem Média Nom"]
    .agg(
        vinculos="count",
        media="mean",
        mediana="median"
    )
)

remuneracao_escolaridade["diferenca_media_mediana"] = (
    remuneracao_escolaridade["media"]
    - remuneracao_escolaridade["mediana"]
)

remuneracao_escolaridade["variacao_mediana_pct"] = (
    remuneracao_escolaridade["mediana"]
    .pct_change()
    * 100
)

remuneracao_escolaridade

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    remuneracao_escolaridade.index.astype(str),
    remuneracao_escolaridade["mediana"],
    marker="o"
)

ax.set_xlabel("Escolaridade")
ax.set_ylabel("Remuneração mediana (R$)")
ax.set_title(
    "Remuneração mediana por nível de escolaridade\n"
    "São Paulo — RAIS 2025"
)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()

plt.savefig(
    FIGURES / "remuneracao_mediana_escolaridade.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Interpretação

A remuneração mediana não apresenta crescimento estritamente monotônico
entre todos os níveis de escolaridade. Nos níveis fundamental e médio
existem oscilações relativamente pequenas e, em alguns casos, reduções
entre categorias consecutivas.

A diferença torna-se mais pronunciada nos níveis mais elevados de
escolaridade. A mediana passa de aproximadamente R$ 2,9 mil entre
vínculos com Superior incompleto para R$ 5,9 mil entre aqueles com
Superior completo.

As maiores medianas são observadas entre vínculos associados a Mestrado
e Doutorado. Esses resultados indicam uma associação positiva mais
acentuada entre escolaridade e remuneração principalmente nos níveis
superiores de formação.

Essas diferenças são descritivas e não devem ser interpretadas como
efeitos causais da escolaridade sobre a remuneração.

In [ ]:
remuneracao_setor_escolaridade = (
    df.groupby(
        ["Setor", "Escolaridade"],
        observed=True
    )["Vl Rem Média Nom"]
    .median()
    .unstack("Escolaridade")
)

remuneracao_setor_escolaridade

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

sns.heatmap(
    remuneracao_setor_escolaridade,
    annot=True,
    fmt=".0f",
    cmap="YlGnBu",
    ax=ax
)

ax.set_xlabel("Escolaridade")
ax.set_ylabel("Setor econômico", labelpad=15)

ax.set_title(
    "Remuneração mediana por escolaridade e setor econômico\n"
    "São Paulo — RAIS 2025",
    pad=15
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0
)

plt.xticks(rotation=45, ha="right")

plt.tight_layout()

plt.savefig(
    FIGURES / "remuneracao_escolaridade_setor.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Interpretação

A comparação entre setores mostra que a remuneração associada aos níveis
de escolaridade não é uniforme entre as atividades econômicas.

As diferenças setoriais são relativamente menores nos níveis mais baixos
de escolaridade, mas tornam-se mais pronunciadas nos níveis superiores.

Entre os vínculos com Superior completo, por exemplo, a Indústria
apresenta remuneração mediana superior à observada no Comércio,
Construção, Agropecuária e Serviços.

O resultado sugere que a associação entre escolaridade e remuneração
varia conforme o setor econômico, hipótese que será posteriormente
investigada por meio da análise estatística.

In [ ]:
contagem_setor_escolaridade = (
    df.groupby(
        ["Setor", "Escolaridade"],
        observed=True
    )
    .size()
    .unstack("Escolaridade")
)

contagem_setor_escolaridade

### Tamanho dos grupos por setor e escolaridade

A quantidade de vínculos varia consideravelmente entre as combinações de
setor e escolaridade. Os níveis de Mestrado e Doutorado são menos
frequentes, especialmente na Agropecuária e na Construção.

Entretanto, em setores como Serviços e Indústria há milhares de vínculos
nessas categorias. Dessa forma, as diferenças observadas entre Mestrado
e Doutorado nesses setores não podem ser atribuídas apenas a grupos
extremamente pequenos.

Ainda assim, diferenças de composição entre ocupações, tipos de vínculo
e outras características dos trabalhadores podem contribuir para os
resultados observados.

## 4. Distribuição da remuneração

A distribuição da remuneração é analisada para verificar sua forma,
dispersão e presença de valores extremos.

Para facilitar a visualização, o histograma é limitado ao percentil 99
da remuneração. Essa limitação é aplicada apenas ao gráfico e não
representa exclusão dessas observações da base analítica.

In [ ]:
p99 = df["Vl Rem Média Nom"].quantile(0.99)

print(f"Percentil 99: R$ {p99:,.2f}")

In [ ]:
remuneracao_visualizacao = df.loc[
    df["Vl Rem Média Nom"] <= p99,
    "Vl Rem Média Nom"
]

media_rem = df["Vl Rem Média Nom"].mean()
mediana_rem = df["Vl Rem Média Nom"].median()

fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(
    remuneracao_visualizacao,
    bins=60,
    ax=ax
)

# Mediana
ax.axvline(
    mediana_rem,
    linestyle="--",
    linewidth=2,
    label=f"Mediana: R$ {mediana_rem:,.2f}"
)

# Média
ax.axvline(
    media_rem,
    linestyle="-.",
    linewidth=2,
    label=f"Média: R$ {media_rem:,.2f}"
)

ax.set_xlabel("Remuneração média nominal (R$)")
ax.set_ylabel("Número de vínculos")

ax.set_title(
    "Distribuição da remuneração média nominal\n"
    "São Paulo — RAIS 2025 (até o percentil 99)"
)

ax.legend()

plt.tight_layout()

plt.savefig(
    FIGURES / "distribuicao_remuneracao.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Interpretação

A distribuição da remuneração apresenta forte assimetria à direita.
A remuneração média é de aproximadamente R$ 4,8 mil, enquanto a
mediana é de aproximadamente R$ 3,0 mil.

O coeficiente de assimetria de 7,70 confirma a presença de uma longa
cauda à direita, causada pela existência de uma parcela relativamente
pequena de vínculos com remunerações muito superiores à maior parte
da população.

O percentil 99 é de aproximadamente R$ 29,8 mil, indicando que 99% dos
vínculos possuem remuneração média nominal abaixo desse valor.

Devido à assimetria da distribuição, a mediana é utilizada como medida
descritiva complementar à média nas comparações salariais.

In [ ]:
media_rem = df["Vl Rem Média Nom"].mean()
mediana_rem = df["Vl Rem Média Nom"].median()
assimetria_rem = df["Vl Rem Média Nom"].skew()

print(f"Média:     R$ {media_rem:,.2f}")
print(f"Mediana:   R$ {mediana_rem:,.2f}")
print(f"Assimetria: {assimetria_rem:.2f}")

In [ ]:
df["Log_Remuneracao"] = np.log(
    df["Vl Rem Média Nom"]
)

fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(
    df["Log_Remuneracao"],
    bins=60,
    ax=ax
)

ax.set_xlabel("Log da remuneração média nominal")
ax.set_ylabel("Número de vínculos")

ax.set_title(
    "Distribuição do log da remuneração média nominal\n"
    "São Paulo — RAIS 2025"
)

plt.tight_layout()

plt.savefig(
    FIGURES / "distribuicao_log_remuneracao.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Transformação logarítmica

A transformação logarítmica reduz substancialmente a assimetria da
distribuição da remuneração, diminuindo a influência visual dos valores
extremamente elevados.

Esse comportamento será considerado na etapa de modelagem estatística,
na qual o logaritmo da remuneração poderá ser utilizado como variável
dependente.

## Síntese da análise exploratória

A análise exploratória revelou quatro características principais da
população estudada.

Primeiro, os vínculos formais estão fortemente concentrados nos níveis
Médio completo e Superior completo, que representam conjuntamente cerca
de 81% da base analítica.

Segundo, a remuneração não cresce de forma estritamente monotônica entre
todos os níveis de escolaridade. As diferenças são relativamente menores
nos níveis fundamental e médio, enquanto aumentos mais expressivos são
observados a partir do Superior completo.

Terceiro, a relação entre escolaridade e remuneração varia entre os
setores econômicos. As diferenças setoriais tornam-se particularmente
visíveis nos níveis mais elevados de escolaridade, indicando que o setor
de atividade pode modificar a associação observada entre formação e
remuneração.

Por fim, a distribuição salarial apresenta forte assimetria à direita,
com média superior à mediana e coeficiente de assimetria de 7,70. A
transformação logarítmica reduz substancialmente essa assimetria e será
considerada na etapa de modelagem estatística.

Os resultados desta etapa são descritivos e indicam associações, não
relações causais. Características como ocupação, idade, jornada de
trabalho, tempo de emprego, tipo de vínculo e tamanho do estabelecimento
também podem estar relacionadas à remuneração.